## <font color='red'> INSTRUCTIONS </font>

<b> 
1. Write your code only in cells below the "WRITE CODE BELOW" title. Do not modify the code below the "DO NOT MODIFY" title. <br>
2. The expected data types of the output answers for each question are given in the last cell through assertion statements. Your answers must match these expected output data types. Hint: Many of the answers need to be a Python dictionary. Consider methods like to_dict() to convert a Pandas Series to a dictionary. <br>
3. The answers are then written to a JSON file named my_results_PA1.json. You can compare this with the provided expected output file "expected_results_PA1.json". <br>
4. After you complete writing your code, click "Kernel -> Restart Kernel and Run All Cells" on the top toolbar. There should NOT be any syntax/runtime errors, otherwise points will be deducted. <br>
5. For submitting your solution, first download your notebook by clicking "File -> Download". Rename the file as &ltTEAM_ID&gt.ipynb" and upload to Canvas.</b>


## <font color='red'> DO NOT MODIFY </font>

In [1]:
import time
import json
import dask
import dask.dataframe as dd
import pandas as pd
import ast
import re
from dask.distributed import Client
import ctypes
import numpy as np

def trim_memory() -> int:
    """
    helps to fix any memory leaks.
    """
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client = Client("127.0.0.1:8786")
client.run(trim_memory)
client = client.restart()
print(client)

None


## <font color='blue'> WRITE CODE BELOW </font>

In [8]:
### read in the user_reviews_path and products_path files, perform your calculations and place the answers in variables ans1 - ans7.
def PA1(user_reviews_path, products_path):

    reviews = dd.read_csv(
        user_reviews_path,
        blocksize="128MB",
        dtype={"asin": "object", "reviewerID": "object"},
        usecols=["reviewerID", "asin", "reviewerName", "helpful", "reviewText",
                 "overall", "summary", "unixReviewTime", "reviewTime"]
    )
    
    products = dd.read_csv(
        products_path,
        blocksize="128MB",
        dtype={"asin": "object"},
        usecols=["asin", "salesRank", "imUrl", "categories", "title",
                 "description", "price", "related", "brand"]
    )
    
    products["price"] = dd.to_numeric(products["price"], errors="coerce")

    # Code to get answers 1 and 2
    ans1_dask = reviews.isna().mean() * 100
    ans2_dask = products.isna().mean() * 100
    
    ans1_result, ans2_result = dask.compute(ans1_dask, ans2_dask)
    
    ans1 = ans1_result.to_dict()
    ans2 = ans2_result.to_dict()

    # Code to get answer 3
    avg_rating = reviews.groupby("asin")["overall"].mean().reset_index()
    price_rating = products[["asin", "price"]].merge(avg_rating, on="asin", how="inner")
    price_rating = price_rating.dropna(subset=["price", "overall"])
    ans3 = float(price_rating["price"].corr(price_rating["overall"]).compute())

    # Code to get answer 4
    price = products["price"].dropna()
    
    price_stats = dask.compute(
        price.mean(),
        price.std(),
        price.quantile(0.5),
        price.min(),
        price.max()
    )
    
    ans4 = {
        "mean": float(price_stats[0]),
        "std": float(price_stats[1]),
        "median": float(price_stats[2]),
        "min": float(price_stats[3]),
        "max": float(price_stats[4]),
    }

    # Code to get answer 5
    def get_super_category(x):
        try:
            cats = ast.literal_eval(x)
            return cats[0][0]
        except Exception:
            return np.nan
    supercats = products["categories"].map(get_super_category, meta=("categories", "object"))
    ans5 = supercats.value_counts().compute().to_dict()

    # Code to get answer 6
    review_asins = reviews[["asin"]].dropna().drop_duplicates()
    product_asins_df = products[["asin"]].dropna().drop_duplicates()
    
    missing_review_asins = review_asins.merge(
        product_asins_df,
        on="asin",
        how="left",
        indicator=True
    )
    
    ans6_count = missing_review_asins[
        missing_review_asins["_merge"] == "left_only"
    ].head(1)
    
    ans6 = int(len(ans6_count) > 0)

    # Code to get answer 7
    product_asins = set(products["asin"].dropna().unique().compute())

    def partition_has_dangling_related(partition, valid_asins):
        for related_str in partition.dropna():
            try:
                related_dict = ast.literal_eval(related_str)
            except Exception:
                continue
    
            if not isinstance(related_dict, dict):
                continue
    
            for value in related_dict.values():
                if isinstance(value, list):
                    for asin in value:
                        if asin not in valid_asins:
                            return True
    
        return False
    
    q7_checks = products["related"].map_partitions(
        partition_has_dangling_related,
        valid_asins=product_asins,
        meta=("has_dangling", "bool")
    )
    
    ans7 = int(q7_checks.any().compute())

    return ans1, ans2, ans3, ans4, ans5, ans6, ans7

## <font color='red'> DO NOT MODIFY </font>

In [ ]:
start = time.time()
ans1, ans2, ans3, ans4, ans5, ans6, ans7 = PA1(user_reviews_path="user_reviews.csv", products_path="products.csv")
end = time.time()
print(f"execution time = {end-start}s")

/home/ubuntu/dask_env/lib/python3.10/site-packages/dask/dataframe/dask_expr/_expr.py:4078: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
/home/ubuntu/dask_env/lib/python3.10/site-packages/dask/dataframe/dask_expr/_expr.py:4078: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
/home/ubuntu/dask_env/lib/python3.10/site-packages/distributed/client.py:3383: UserWarning: Sending large graph of size 117.00 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into t

In [ ]:
# DO NOT MODIFY
assert type(ans1) == dict, f"answer to question 1 must be a dictionary like {{'reviewerID':0.2, ..}}, got type = {type(ans1)}"
assert type(ans2) == dict, f"answer to question 2 must be a dictionary like {{'asin':0.2, ..}}, got type = {type(ans2)}"
assert type(ans3) == float, f"answer to question 3 must be a float like 0.8, got type = {type(ans3)}"
assert type(ans4) == dict, f"answer to question 4 must be a dictionary like {{'mean':0.4,'max':0.6,'median':0.6...}}, got type = {type(ans4)}"
assert type(ans5) == dict, f"answer to question 5 must be a dictionary, got type = {type(ans5)}"         
assert ans6 == 0 or ans6==1, f"answer to question 6 must be 0 or 1, got value = {ans6}" 
assert ans7 == 0 or ans7==1, f"answer to question 7 must be 0 or 1, got value = {ans7}" 

ans_dict = {
    "q1": ans1,
    "q2": ans2,
    "q3": ans3,
    "q4": ans4,
    "q5": ans5,
    "q6": ans6,
    "q7": ans7,
    "runtime": end-start
}
with open('my_results_PA1.json', 'w') as outfile: json.dump(ans_dict, outfile)         